# ComfyUI TPU Experimental — Colab

> **實驗性 Notebook；尚未於 TPU 上驗證完整 ComfyUI 圖片生成。** 預設 GPU，不會假裝 TPU 相容。

- Colab → 執行階段 → 變更執行階段類型 → TPU (TPU 模式) 或 GPU (GPU 模式)。
- TPU 路線使用 [ComfyUI-XLA](https://github.com/radna0/ComfyUI-XLA) 的舊版 PyTorch/XLA 2.5；當前執行階段不一定相容。
- 不自動下載模型、不公開未驗證的 ComfyUI 服務。此 Notebook 不提供公開 tunnel，防止任意節點被網際網路訪客執行。
- TPU smoke test、服務啟動及真正推論是不同的驗證階段。

In [ ]:
import sys, os, pathlib, subprocess, importlib.util
BASE = pathlib.Path("/content")
BASE.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version)
print("Base:", BASE)
print("GPU:", subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() or "Not detected")
print("torch_xla present:", bool(importlib.util.find_spec("torch_xla")))

## 1. 選擇執行模式

TPU 模式必須有 TPU 執行階段及相容的 Python 3.10 環境。不要在 Python 不相容時強行降級系統 PyTorch。

In [ ]:
MODE = "gpu"  # "gpu" or "tpu_experimental"
assert MODE in ("gpu", "tpu_experimental")
if MODE == "tpu_experimental":
    if sys.version_info[:2] != (3, 10):
        raise RuntimeError("ComfyUI-XLA 的舊版相依套件以 Python 3.10 為基準；目前 Python 版本不相容，請換相容環境。")
    os.environ["PJRT_DEVICE"] = "TPU"
else:
    p = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError("GPU 模式需要 NVIDIA GPU。請在執行階段設定選擇 GPU。")

## 2. 取得 ComfyUI 原始碼

GPU 使用官方 ComfyUI；TPU 使用實驗性 ComfyUI-XLA fork。

In [ ]:
URL = "https://github.com/radna0/ComfyUI-XLA.git" if MODE == "tpu_experimental" else "https://github.com/Comfy-Org/ComfyUI.git"
APP = BASE / ("ComfyUI_TPU_XLA" if MODE == "tpu_experimental" else "ComfyUI_GPU")
if not APP.exists():
    subprocess.run(["git", "clone", "--depth", "1", URL, str(APP)], check=True)
print(APP, URL)

## 3. TPU 相依套件／裝置 smoke test

舊版 `torch_xla` 與現行執行階段可能衝突；安裝後如版本被替換，需要重新啟動執行階段。

In [ ]:
if MODE == "tpu_experimental":
    subprocess.run([sys.executable, "-m", "pip", "install", "torch~=2.5.0", "torch_xla[tpu]~=2.5.0", "-f", "https://storage.googleapis.com/libtpu-releases/index.html"], check=True)
    print("TPU 套件安裝完成；如修改了已載入的 torch，請重新啟動執行階段，再依序執行。")
else:
    print("GPU 模式保留執行階段提供的 PyTorch/CUDA。")

In [ ]:
import torch
if MODE == "tpu_experimental":
    import torch_xla.core.xla_model as xm
    if not torch.__version__.split("+")[0].startswith("2.5."):
        raise RuntimeError("此 ComfyUI-XLA 實驗路線要求 torch 2.5.x。")
    device = xm.xla_device()
    a = torch.ones((32, 32), device=device)
    result = a @ a
    xm.mark_step()
    print("TPU smoke test:", device, result.cpu()[0, 0].item())
else:
    assert torch.cuda.is_available(), "CUDA unavailable"
    print("GPU:", torch.cuda.get_device_name(0))

## 4. 安裝其餘依賴

排除 torch、torchvision、torchaudio、xformers，避免覆蓋已選好的加速器套件。

In [ ]:
req_file = APP / "requirements.txt"
if not req_file.exists():
    raise FileNotFoundError(f"Missing requirements: {req_file}")
reqs = []
for raw in req_file.read_text().splitlines():
    ln = raw.strip()
    if not ln or ln.startswith("#"):
        continue
    pkg = ln.lower().replace("_", "-")
    if pkg.startswith(("torch", "xformers")):
        continue
    reqs.append(ln)
filtered = APP / "requirements-no-torch.txt"
filtered.write_text("\n".join(reqs) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(filtered)], check=True)

## 5. 啟動本機服務

預設只監聽 `127.0.0.1`。Notebook 中的服務不會自動提供外部公開網址。使用遠端瀏覽器前，需另行設定有身分驗證的存取管道。

In [ ]:
import time
log_path = APP / "comfyui.log"
cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"]
if MODE == "tpu_experimental":
    cmd.append("--xla")
log_handle = log_path.open("a")
server = subprocess.Popen(cmd, cwd=str(APP), stdout=log_handle, stderr=subprocess.STDOUT, env=os.environ.copy())
time.sleep(8)
print("PID:", server.pid, "exit:", server.poll())
print(log_path.read_text(errors="replace")[-4000:])

## 6. 模型配置與限制

將合法取得且格式相容的模型放入 `APP/models/checkpoints`。請自行測試 checkpoint、CLIP、KSampler、VAE decode 與推論輸出。TPU 基礎矩陣運算成功或 ComfyUI 啟動成功都**不代表**完整 workflow 可在 TPU 上執行。

參考：[ComfyUI-XLA](https://github.com/radna0/ComfyUI-XLA) · [PyTorch/XLA](https://github.com/pytorch/xla) · [ComfyUI](https://github.com/Comfy-Org/ComfyUI)

In [ ]:
print("Checkpoints:", APP / "models" / "checkpoints")
print("Server:", "running" if server.poll() is None else f"exited ({server.returncode})")
print(log_path.read_text(errors="replace")[-5000:])